# STEP 04C — Repair failed Fold 1 only (resume, no retraining)

**What happened:** the previous Step 04C completed 19/20 fold evaluations, but a YOLO11s Fold 1 prediction for `cate1_013.png` had non-positive width or height after clipping to image boundaries. The previous safety assertion correctly stopped that fold. The subsequent summary assertion therefore correctly refused to analyze 19/20 results.

**What this notebook does:**
1. Reuses the exact completed 19 folds from the prior run in Drive, without re-evaluating them.
2. Verifies all 20 checkpoint SHA-256 hashes and the 70 fold validation image/label input hashes.
3. Runs **only YOLO11s Fold 1**, explicitly documents and discards zero-area predictions after clipping, and computes COCO AP with the same 0.001 threshold / max 100 predictions / IoU settings.
4. Writes a **new immutable evidence output folder**, preserving the earlier incomplete run, recomputes mean/SD, Friedman and Holm-adjusted paired Wilcoxon tests from the now-complete 20 rows, and produces a ZIP.

**Important:** Removing geometrically invalid predictions is a documented *technical eligibility filter*, not manual adjustment based on ground truth or success/failure. All exclusions are recorded in `dropped_invalid_prediction_boxes.csv`, including source image, confidence, raw and clipped coordinates. No source labels or model weights are modified. This control still harmonizes only the evaluator, **not** model training.

Run all cells in Google Colab with a CUDA GPU; if your prior run folder has a different name, edit `SOURCE_NAME` in Cell 2 only.


In [ ]:
%pip -q install "ultralytics==8.4.138" "torchmetrics>=1.3,<2" pycocotools scipy pandas pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os,sys,json,time,hashlib,platform,traceback,zipfile,gc,shutil
from pathlib import Path
from datetime import datetime,timezone
from itertools import combinations
import numpy as np
import pandas as pd
import torch,torchvision,ultralytics,torchmetrics
from PIL import Image
from ultralytics import YOLO,RTDETR
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from scipy.stats import friedmanchisquare,wilcoxon
from torchvision.models.detection import fasterrcnn_resnet50_fpn,retinanet_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.retinanet import RetinaNetClassificationHead

assert torch.cuda.is_available(), 'Enable GPU: Runtime > Change runtime type > T4 GPU'
ROOT=Path('/content/drive/MyDrive/Pediatric_Caries_AI')
CV=ROOT/'cross_validation'
SOURCE_NAME='Step_04C_Unified_COCO_AP_20261024T174836Z'  # Previous 19/20 run
SOURCE=ROOT/'reviewer_revision'/SOURCE_NAME
assert CV.is_dir(),f'Missing project: {CV}'
assert SOURCE.is_dir(),f'Missing partial output folder: {SOURCE}'
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT=ROOT/'reviewer_revision'/f'Step_04C_Repair_Resume_{RUN_ID}'
OUT.mkdir(parents=True,exist_ok=False)
DEV='cuda:0'; SEED=42; IMG_SIZE=1024; SCORE_FLOOR=.001; NMS_IOU=.70; MAX_DET=100
FAMILIES=['YOLO11s','RT-DETR-L','Faster R-CNN','RetinaNet']
EXPECTED_FOLD_GT=[50,48,31,49,28]
np.random.seed(SEED);torch.manual_seed(SEED)
print('CUDA:',torch.cuda.get_device_name(0))
print('Existing partial run:',SOURCE)
print('New results folder:',OUT)
print('Library versions:',torch.__version__,torchvision.__version__,ultralytics.__version__,torchmetrics.__version__)
assert ultralytics.__version__=='8.4.138','Ultralytics version changed; stop and review.'
previous_env=json.loads((SOURCE/'run_environment_and_protocol.json').read_text())
for lib, current in [('torch',torch.__version__),('torchvision',torchvision.__version__),
                     ('ultralytics',ultralytics.__version__),('torchmetrics',torchmetrics.__version__)]:
 assert current==previous_env[lib],f'{lib} version mismatch: old={previous_env[lib]}, now={current}; do not mix AP results' 

# Check that this is precisely the known one-failure partial run.
partial=pd.read_csv(SOURCE/'unified_coco_per_fold.csv')
assert len(partial)==20,(len(partial),'Expected 20 rows')
assert partial[['family','fold']].drop_duplicates().shape[0]==20,'Duplicate/missing family-fold rows'
assert set(partial['family'])==set(FAMILIES)
bad=partial.loc[partial['status']!='OK',['family','fold','status','error']]
print('Failed rows in archived partial run:')
print(bad.to_string(index=False))
assert len(bad)==1 and bad.iloc[0]['family']=='YOLO11s' and int(bad.iloc[0]['fold'])==1,        'Stop: archive has unexpected failures. Do not resume blindly.'
for _,r in partial.loc[partial.status=='OK'].iterrows():
 p=SOURCE/f"predictions_{r['family'].replace(' ','_')}_fold{int(r['fold'])}.csv"
 assert p.is_file(),f'Missing completed-fold prediction export: {p}'
 t=pd.read_csv(p)
 count=int((t['prediction_index']>=0).sum())
 assert count==int(r['n_predictions']),('Inconsistent archived prediction count',p,count,r['n_predictions'])

# Preserve the old incomplete run, copy its evidence into a new output directory.
for p in SOURCE.iterdir():
 if p.is_file() and p.suffix.lower()!='.zip':
  shutil.copy2(p,OUT/p.name)
(OUT/'SOURCE_RUN_REFERENCE.txt').write_text(str(SOURCE)+'\nPrevious 19 OK/1 ERROR evidence copied unchanged except the one repaired fold metrics/output.\n')
print('Verified 19 previous folds and copied their artifacts without recomputing them.')


In [ ]:
# Fail before evaluation if any checkpoint has changed since the successful 04B provenance run.
EXPECTED_HASHES = {'YOLO11s|1': '0c8be9879987a156c08801fe7bcd72236010522b322af3bfe1796cd6a4ca4a58', 'RT-DETR-L|1': '22575dba1a9f5d32e37f3feb7a1389693223479ec88d81169c12d8f9deeca222', 'Faster R-CNN|1': 'fe6c3b4165bdfe872519a5ece501118846326050ccf1201fda68acff1303b084', 'RetinaNet|1': '7d73dc6177e5c19c1a59f02f0ea2c437e948043e2f378c6b0723405bf35649af', 'YOLO11s|2': '701c0e3841521c9ba6cd68a45bd86f40ac6e670bd7afb5b60bf8e321c3969403', 'RT-DETR-L|2': 'e6046bd4e64e4975ee8f7296fb30bbd724b481a88eb470f31e17ca702e7a9492', 'Faster R-CNN|2': '603bce3bdeca056834e3bd9e3f1d4885bb278a32ac06f054ea0d63ddfe288428', 'RetinaNet|2': 'f180d660f794ee45bb0bb672956c2126a93dc94bbe2e4dd47be9d642e0666c8f', 'YOLO11s|3': '5f45dd73709ebd7907db56ba1dcf9db294b0b71579f0e0486cb9a8b4b7f8e726', 'RT-DETR-L|3': '487f305c2fa3d372c5ce5b86c2f85f38ae8604d5cbfe8aaf213d1e4df4115192', 'Faster R-CNN|3': '9251577be1af6360302429210b13ff7611d6f7b26be9ba7ac37b0da5f2ea16e5', 'RetinaNet|3': '268e21a878b8db6f5da9c6721f6b7a91a354a8c9ddb10997367cbecad2d70ba1', 'YOLO11s|4': 'b280e7f979f1148dc04ca5a60138f3d60bebaed38991c92326d0e5c6ae601d27', 'RT-DETR-L|4': '17f4d986a8fa1c20dfb5c20b5bbc3cc7809fbbca73d19f2e36603910949f30f6', 'Faster R-CNN|4': 'd4801839dd7e2d691174e152317cfc6c6c4ceab08faa9d2cec427657a2724bf6', 'RetinaNet|4': '7a0ecc488c04f2175b9bbbb8a1ad183ad3e1fc79f64bce54a4b032129d3422ca', 'YOLO11s|5': '998e555d607e1203991578589d57e0c9369e13f2de6d3f116356e3e169441cb9', 'RT-DETR-L|5': 'd75949e566f5df8413498879a9576069530f0239e34bc1310cb6b173f7461a82', 'Faster R-CNN|5': '63ed7946ab05c0dc53708bcd0fabbaa2bf2b7d518c0281605eae807a8b44527f', 'RetinaNet|5': '0000c5eec1021cd575df66430e2e1f484737bc17ae1667812ee4460bb21666ca'}

def sha256(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(4*1024*1024),b''): h.update(b)
 return h.hexdigest()

def ckpt(fam,fold):
 if fam=='YOLO11s':return CV/f'models/YOLO11s/fold_{fold}/weights/best.pt'
 if fam=='RT-DETR-L':return CV/f'models/RTDETR/fold_{fold}/weights/best.pt'
 if fam=='Faster R-CNN':return CV/f'models/FasterRCNN/fold_{fold}/best.pt'
 if fam=='RetinaNet':return CV/f'models/RetinaNet/fold_{fold}/best.pt'
 raise ValueError(fam)

checks=[]
for fam in FAMILIES:
 for fold in range(1,6):
  p=ckpt(fam,fold)
  assert p.exists(),f'Missing: {p}'
  h=sha256(p)
  assert h==EXPECTED_HASHES[f'{fam}|{fold}'],f'CHECKPOINT CHANGED: {fam} fold={fold}: {h}'
  checks.append({'family':fam,'fold':fold,'relative_path':str(p.relative_to(ROOT)),'sha256':h,'size_bytes':p.stat().st_size})
pd.DataFrame(checks).to_csv(OUT/'checkpoint_sha256_verified.csv',index=False)
print('PASS: All 20 checkpoint hashes match Step 04B.')

In [ ]:
# Shared GT reader: same 14 images and YOLO normalized caries boxes for all four pipelines.
IMAGE_SUFFIXES={'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}
def fold_paths(fold):
 base=CV/f'fold_{fold}'
 images=sorted([p for p in (base/'images/val').iterdir() if p.suffix.lower() in IMAGE_SUFFIXES])
 assert len(images)==14,(fold,len(images))
 labeldir=base/'labels/val'
 return images,labeldir

def read_gt(imgpath,labeldir):
 with Image.open(imgpath) as im:
  W,H=im.size
 lp=labeldir/(imgpath.stem+'.txt')
 assert lp.is_file(),lp
 lines=[l.strip() for l in lp.read_text().splitlines() if l.strip()]
 boxes=[]
 for line in lines:
  fields=line.split()
  assert len(fields)==5,(lp,line)
  c,x,y,w,h=map(float,fields)
  assert int(c)==0 and all(np.isfinite([x,y,w,h])) and 0<=x<=1 and 0<=y<=1 and 0<w<=1 and 0<h<=1,(lp,line)
  xx1=(x-w/2)*W; yy1=(y-h/2)*H; xx2=(x+w/2)*W; yy2=(y+h/2)*H
  assert xx2>xx1 and yy2>yy1 and xx1>=-0.15 and yy1>=-0.15 and xx2<=W+0.15 and yy2<=H+0.15,(lp,line)
  boxes.append([max(0.,xx1),max(0.,yy1),min(float(W),xx2),min(float(H),yy2)])
 arr=torch.tensor(boxes,dtype=torch.float32).reshape(-1,4)
 return {'boxes':arr,'labels':torch.zeros((len(arr),),dtype=torch.int64)}, W,H,lp

man=[]
for f in range(1,6):
 images,labels=fold_paths(f)
 ng=0
 for ip in images:
  gt,W,H,lp=read_gt(ip,labels)
  ng+=len(gt['boxes'])
  man.append({'fold':f,'image':ip.name,'W':W,'H':H,'n_gt':len(gt['boxes']),
             'image_sha256':sha256(ip),'label_sha256':sha256(lp)})
 assert ng==EXPECTED_FOLD_GT[f-1],f'Fold {f}: expected {EXPECTED_FOLD_GT[f-1]} lesions, found {ng}'
assert len(man)==70
pd.DataFrame(man).to_csv(OUT/'fold_gt_manifest_sha256.csv',index=False)
print('PASS: 70 held-out validation image instances, 206 source-defined caries boxes.')# Compare input hashes to the prior Step 04C run manifest, not only counts.
prev_manifest=pd.read_csv(SOURCE/'fold_gt_manifest_sha256.csv')
new_manifest=pd.read_csv(OUT/'fold_gt_manifest_sha256.csv')
cols=['fold','image','image_sha256','label_sha256','n_gt','W','H']
joined=new_manifest[cols].merge(prev_manifest[cols],on=['fold','image'],how='outer',suffixes=('_now','_before'),indicator=True)
assert (joined['_merge']=='both').all() and len(joined)==70
for col in ['image_sha256','label_sha256','n_gt','W','H']:
 assert (joined[col+'_now']==joined[col+'_before']).all(),f'Input mismatch in {col}'
print('PASS: source image and label SHA-256 stable for all 70 fold instances.')


In [ ]:
# Prediction generators. Filter and LOG geometrically invalid clipped boxes; never optimize by GT.
DROPPED_AUDIT=[]
def build_torchvision_model(fam):
 if fam=='Faster R-CNN':
  m=fasterrcnn_resnet50_fpn(weights=None,weights_backbone=None)
  n=m.roi_heads.box_predictor.cls_score.in_features
  m.roi_heads.box_predictor=FastRCNNPredictor(n,2)
  m.roi_heads.score_thresh=SCORE_FLOOR
  m.roi_heads.nms_thresh=NMS_IOU
  m.roi_heads.detections_per_img=MAX_DET
 elif fam=='RetinaNet':
  m=retinanet_resnet50_fpn(weights=None,weights_backbone=None)
  m.head.classification_head=RetinaNetClassificationHead(m.backbone.out_channels,m.head.classification_head.num_anchors,2)
  m.score_thresh=SCORE_FLOOR
  m.nms_thresh=NMS_IOU
  m.detections_per_img=MAX_DET
 else: raise ValueError(fam)
 return m

def load_model(fam,fold):
 p=ckpt(fam,fold)
 if fam in ['YOLO11s','RT-DETR-L']:
  model=(YOLO if fam=='YOLO11s' else RTDETR)(str(p))
  assert model.names=={0:'caries'},(fam,model.names)
  return model
 m=build_torchvision_model(fam)
 state=torch.load(str(p),map_location='cpu',weights_only=True)
 if isinstance(state,dict) and 'state_dict' in state:state=state['state_dict']
 m.load_state_dict(state,strict=True)
 del state
 return m.eval().to(DEV)

@torch.inference_mode()
def predict_one(fam,model,ip,W,H):
 if fam in ['YOLO11s','RT-DETR-L']:
  results=model.predict(source=str(ip),device=0,imgsz=IMG_SIZE,conf=SCORE_FLOOR,
                         iou=NMS_IOU,max_det=MAX_DET,rect=False,half=False,
                         augment=False,save=False,verbose=False,stream=False)
  assert len(results)==1,(fam,ip,len(results))
  bb=results[0].boxes
  if bb is None:
   box=torch.empty((0,4),dtype=torch.float32)
   scores=torch.empty((0,),dtype=torch.float32)
   labels=torch.empty((0,),dtype=torch.long)
  else:
   box=bb.xyxy.detach().cpu().to(torch.float32)
   scores=bb.conf.detach().cpu().to(torch.float32)
   labels=bb.cls.detach().cpu().to(torch.int64)
 else:
  im=Image.open(ip).convert('RGB')
  tens=torchvision.transforms.functional.to_tensor(im).to(DEV)
  im.close()
  out=model([tens])[0]
  box=out['boxes'].detach().cpu().to(torch.float32)
  scores=out['scores'].detach().cpu().to(torch.float32)
  labels=out['labels'].detach().cpu().to(torch.int64)
  assert torch.all(labels==1),(fam,ip,torch.unique(labels))
  labels=torch.zeros_like(labels)
 # Require predictable clean and standardized outputs (no silent discarding).
 assert box.ndim==2 and box.shape[1]==4 and len(box)==len(scores)==len(labels)
 assert torch.isfinite(box).all() and torch.isfinite(scores).all()
 assert torch.all(labels==0),(fam,ip,torch.unique(labels))
 inds=(scores>=SCORE_FLOOR).nonzero().reshape(-1)
 inds=inds[torch.argsort(scores[inds],descending=True,stable=True)][:MAX_DET]
 box=box[inds].clone();scores=scores[inds].clone();labels=labels[inds].clone()
 raw=box.clone()
 box[:,[0,2]]=box[:,[0,2]].clamp(0,W)
 box[:,[1,3]]=box[:,[1,3]].clamp(0,H)
 # COCO bbox metrics require positive width and height. Log EVERY invalid box,
 # then omit it. This decision depends exclusively on prediction geometry.
 valid=(box[:,2]>box[:,0]) & (box[:,3]>box[:,1])
 for j in (~valid).nonzero().reshape(-1).tolist():
  rb=raw[j].tolist();cb=box[j].tolist()
  reason=('nonpositive_width_and_height' if cb[2]<=cb[0] and cb[3]<=cb[1] else
          'nonpositive_width' if cb[2]<=cb[0] else 'nonpositive_height')
  DROPPED_AUDIT.append({'family':fam,'fold':1,'image':ip.name,
    'original_prediction_index':int(inds[j]),'confidence':float(scores[j]),
    'x1_raw':rb[0],'y1_raw':rb[1],'x2_raw':rb[2],'y2_raw':rb[3],
    'x1_clipped':cb[0],'y1_clipped':cb[1],'x2_clipped':cb[2],'y2_clipped':cb[3],
    'reason':reason})
 box=box[valid];scores=scores[valid];labels=labels[valid]
 assert torch.all(box[:,2]>box[:,0]) and torch.all(box[:,3]>box[:,1])
 return {'boxes':box,'scores':scores,'labels':labels}

print('Configured common COCO evaluator and model-specific output adapters.')

In [ ]:
# Re-evaluate ONLY YOLO11s Fold 1. Other 19 folds were audited and copied intact.
family='YOLO11s'; fold=1
started=time.time(); ims,labdir=fold_paths(fold)
model=load_model(family,fold)
metric=MeanAveragePrecision(box_format='xyxy',iou_type='bbox',backend='pycocotools',
                            max_detection_thresholds=[1,10,100],class_metrics=False)
pr=[];num_gt=0;num_predictions=0
for ip in ims:
 gt,W,H,lp=read_gt(ip,labdir)
 pred=predict_one(family,model,ip,W,H)
 metric.update([pred],[gt])
 num_gt+=len(gt['boxes']);num_predictions+=len(pred['boxes'])
 if len(pred['boxes'])==0:
  pr.append({'family':family,'fold':fold,'image':ip.name,'prediction_index':-1,
             'x1':np.nan,'y1':np.nan,'x2':np.nan,'y2':np.nan,'score':np.nan,'label':np.nan})
 for j,(box,score,label) in enumerate(zip(pred['boxes'].tolist(),pred['scores'].tolist(),pred['labels'].tolist())):
  pr.append({'family':family,'fold':fold,'image':ip.name,'prediction_index':j,
             'x1':box[0],'y1':box[1],'x2':box[2],'y2':box[3],
             'score':float(score),'label':int(label)})
assert num_gt==50
rr=metric.compute()
vals={k:float(rr[k]) for k in ['map_50','map','mar_100']}
assert all(0<=x<=1 for x in vals.values()),vals
pd.DataFrame(pr).to_csv(OUT/'predictions_YOLO11s_fold1.csv',index=False)
columns=['family','fold','image','original_prediction_index','confidence',
         'x1_raw','y1_raw','x2_raw','y2_raw','x1_clipped','y1_clipped','x2_clipped','y2_clipped','reason']
pd.DataFrame(DROPPED_AUDIT,columns=columns).to_csv(OUT/'dropped_invalid_prediction_boxes.csv',index=False)
old_row=partial.loc[(partial.family==family)&(partial.fold==fold)]
assert len(old_row)==1
updated=partial.copy()
pos=(updated.family==family)&(updated.fold==fold)
for key,val in {'COCO_mAP50':vals['map_50'],'COCO_mAP50_95':vals['map'],
                'COCO_mAR100':vals['mar_100'],'n_images':14,'n_gt':50,
                'n_predictions':num_predictions,'status':'OK','error':'',
                'duration_seconds':round(time.time()-started,1)}.items():
 updated.loc[pos,key]=val
assert len(updated)==20 and (updated.status=='OK').all(),'Incomplete or failed folds remain'
updated.to_csv(OUT/'unified_coco_per_fold.csv',index=False)
print('PASS repaired YOLO11s Fold 1:',vals)
print('Prediction boxes retained:',num_predictions,'invalid boxes logged and excluded:',len(DROPPED_AUDIT))
print('Invalid boxes per image:',pd.DataFrame(DROPPED_AUDIT).image.value_counts().to_dict() if DROPPED_AUDIT else {})
print('Completed:',len(updated),'folds, all OK. Previous 19 folds were not rerun.')
if len(DROPPED_AUDIT)>10:
 raise RuntimeError('More than 10 invalid boxes in one fold: inspect dropped_invalid_prediction_boxes.csv before interpreting these numbers.')
del model,metric
gc.collect();torch.cuda.empty_cache()


In [ ]:
# A secondary statistical summary from standardized-AP fold-wise results.
metrics=pd.read_csv(OUT/'unified_coco_per_fold.csv')
assert len(metrics)==20 and (metrics.status=='OK').all(), 'STOP: some folds failed; do not infer full results from partial data.'
summary=[];testrows=[];pairrows=[]
for m in ['COCO_mAP50','COCO_mAP50_95']:
 wide=metrics.pivot(index='fold',columns='family',values=m)[FAMILIES]
 assert wide.shape==(5,4) and not wide.isna().any().any()
 for fam in FAMILIES:
  y=wide[fam]
  summary.append({'model':fam,'metric':m,'n_folds':5,'mean':y.mean(),'sample_SD':y.std(ddof=1),
                  'min':y.min(),'max':y.max()})
 fr=friedmanchisquare(*(wide[fam] for fam in FAMILIES))
 testrows.append({'metric':m,'friedman_chi2':float(fr.statistic),
                  'friedman_p_asymptotic':float(fr.pvalue),'n_folds':5})
 local=[]
 for a,b in combinations(FAMILIES,2):
  w=wilcoxon(wide[a],wide[b],alternative='two-sided',zero_method='wilcox',method='auto')
  local.append({'metric':m,'model_a':a,'model_b':b,'W':float(w.statistic),'p_raw':float(w.pvalue)})
 # Holm step-down in predeclared 6 tests for each endpoint; enforce monotonicity.
 ordered=sorted(range(len(local)),key=lambda k:local[k]['p_raw'])
 last=0.
 for rank,idx in enumerate(ordered):
  candidate=min(1.,(len(local)-rank)*local[idx]['p_raw'])
  last=max(last,candidate)
  local[idx]['p_Holm']=last
  local[idx]['significant_at_0_05']=bool(last<.05)
 pairrows+=local
pd.DataFrame(summary).to_csv(OUT/'unified_coco_mean_sd.csv',index=False)
pd.DataFrame(testrows).to_csv(OUT/'unified_coco_friedman.csv',index=False)
pd.DataFrame(pairrows).to_csv(OUT/'unified_coco_wilcoxon_holm.csv',index=False)
print(pd.DataFrame(summary).to_string(index=False))
print(pd.DataFrame(testrows).to_string(index=False))
print('Pairwise Holm-adjusted significant:',sum(x['significant_at_0_05'] for x in pairrows))

In [ ]:
# Reproducibility audit summary + shareable ZIP.
metrics=pd.read_csv(OUT/'unified_coco_per_fold.csv')
assert len(metrics)==20 and (metrics.status=='OK').all(), 'Refuse partial analyses'
assert all((OUT/f'predictions_{f.replace(" ","_")}_fold{k}.csv').is_file() for f in FAMILIES for k in range(1,6))
meta=json.loads((OUT/'run_environment_and_protocol.json').read_text())
meta['repair_run_utc']=RUN_ID
meta['repair_source_dir']=str(SOURCE)
meta['repair_scope']='ONLY YOLO11s Fold 1, repair of zero-area clipped prediction bounding boxes'
meta['repair_filter']='after clipping to [0,W] x [0,H], discard only predictions with x2<=x1 or y2<=y1; log each discarded prediction'
meta['repair_dropped_count']=len(DROPPED_AUDIT)
meta['restored_previous_folds_without_recalculation']=19
(OUT/'run_environment_and_protocol.json').write_text(json.dumps(meta,indent=2))
readme="""STEP 04C / REPAIRED RESUME EVIDENCE BUNDLE

Source incomplete run: {src}
Successfully reused 19 folds; only YOLO11s fold 1 was recomputed.
The prior run stopped because its pre-metric assertion found a prediction box with
non-positive width or height after clipping to image boundaries on cate1_013.png.
The new rule excludes such invalid detection boxes, logs them all, and does not
consider ground-truth overlap in deciding what to drop.
Discarded count in the rerun: {n}
The raw input images and original GT remain unchanged; all 20 weights and all
70 validation image-label pairs were SHA-256 verified.
COCO-style mAP uses TorchMetrics/pycocotools, confidence floor 0.001, at most
100 predictions/image, same 14 validation images/fold.
The 19 original fold runs did not trigger a geometrical validity error.
Evaluation is standardized, but preprocessing/training budgets are not.
These results are a secondary metric harmonization analysis, not a controlled
architecture-only comparison. Five overlapping CV folds limit inferential power.
""".format(src=SOURCE,n=len(DROPPED_AUDIT))
(OUT/'README_REPAIR_AND_LIMITATIONS.txt').write_text(readme)
archive=OUT/'Step_04C_Unified_COCO_AP_REPAIRED_Results.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
 for p in sorted(OUT.iterdir()):
  if p.is_file() and p!=archive: z.write(p,p.name)
print('SUCCESS: 20/20 folds OK.')
print('Corrected result archive:',archive)
print('ZIP size:',archive.stat().st_size,'bytes')
print('Upload this ZIP to ChatGPT. Do not edit the manuscript before auditing its numbers.')
